In [1]:
import sqlite3
conn = sqlite3.connect("../outputs/hpo/fixed_hpo/optuna.db")

In [5]:
import json
import pandas as pd

# Recompute the grouped z-scores
with open("./baseline_metrics.json", "r") as f:
    baseline_metrics = json.load(f)

with open("../outputs/hpo/fixed_hpo/outputs.json", "r") as f:
    hpo_data = json.load(f)

In [13]:
print(hpo_data["1"]["bend_benchmark"])

{'gene_finding': [{'val/mcc': 0.1122389411478369, 'val/loss': 1.2978233098983765}], 'cpg_methylation': [{'val/mcc': 0.1122389411478369, 'val/loss': 1.2978233098983765}]}


In [15]:
with open("../outputs/hpo/fixed_hpo/outputs.json", "w") as f:
    json.dump(hpo_data, f)

In [10]:
trial_scores = {}
for trial in hpo_data.keys():
    trial_scores[trial] = {}

for dataset_name in ("nt_benchmark", "genomic_benchmark", "bend_benchmark", "chromatin_profile_prediction"):
    def get_tasks(trial):
        global dataset_name
        if dataset_name == "chromatin_profile_prediction":
            cpp = trial.get("chromatin_profile_prediction")
            if cpp is not None:
                return {"chromatin_profile_prediction": cpp}
            else:
                return None
        else:
            return trial.get(dataset_name)

    for trial in hpo_data.keys():
        tasks = get_tasks(hpo_data[trial])

        if tasks is not None:
            for name, val in tasks.items():
                if name == "pretraining": 
                    continue

                try:
                    trial_scores[trial][name] = val[0][baseline_metrics[name]["metric"]]
                except Exception as e:
                    print(trial_scores[trial])
                    raise e

{'splice_sites_acceptors': 0.5414511070332801, 'H3K27me3': 0.5055646241387338, 'splice_sites_all': 0.35036603212659984, 'H3K4me2': 0.46992066168314983, 'H2AFZ': 0.385682920663283, 'H3K4me1': 0.40496243378212604, 'H3K27ac': 0.44319792090572757, 'H3K9me3': 0.2833965510330238, 'H3K4me3': 0.6190372171099232, 'enhancers_types': 0.4449056048664632, 'promoter_all': 0.7083859101823108, 'enhancers': 0.46705562972134945, 'promoter_tata': 0.7082005278281703, 'splice_sites_donors': 0.5764074491577298, 'H3K36me3': 0.5188005612618146, 'H4K20me1': 0.5705181380595311, 'promoter_no_tata': 0.7406087272553363, 'H3K9ac': 0.5112271264019361, 'demo_coding_vs_intergenomic_seqs': 0.9324399828910828, 'human_nontata_promoters': 0.9145450592041016, 'drosophila_enhancers_stark': 0.7213872671127319, 'dummy_mouse_enhancers_ensembl': 0.8016529083251953, 'human_enhancers_cohn': 0.7402130365371704, 'demo_human_or_worm': 0.9670000076293945, 'human_enhancers_ensembl': 0.877236008644104, 'human_ocr_ensembl': 0.7862496972

KeyError: 'val/auroc'